In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
import jax.numpy as jnp
from tqdm import tqdm
from lunanav.constants import *
from lunanav.sim.simulator import SimParams, RigidBody, SimResults
from lunanav.sim.sensors import (
    SensorSuite,
    accelerometer_sensor, gyroscope_sensor,
    laser_altimeter_sensor, laser_velocity_sensor,
    star_tracker_sensor, doppler_sensor, sat_range_tracker_sensor
)
from lunanav.sim.generate import SatPosVel, make_sat_arrs, generate_env, generate_measurements
from lunanav.estimation.ekf import ekf_predict, Qd_from_accel_white, update_sensor, update_sensor_individual_NaN_check
from lunanav.sim.quaternion import unitize_state
from lunanav.loaders import load_sim_result

In [ ]:
sim_save_path = "traj_B_liftoff.json"
sim_result = load_sim_result(f"data/simresults/{sim_save_path}")

dt      = sim_result.dt
n       = sim_result.nsteps
t_max   = dt * n

results = SimResults(n)
results.t        = sim_result.t
results.states   = sim_result.s_arr
results.force_N  = sim_result.force
results.torque_Nm = sim_result.torque
results.nsteps   = n

lander = RigidBody(mass_kg=sim_result.mass_kg, I=sim_result.I)
sim    = SimParams(results.states[0], lander, dt, t_max)

measurements_noisy = {k: np.array(v["noisy"]) for k, v in sim_result.measurements.items()}

print(f"Loaded: {n} steps, dt={dt}s")
print(f"Max force:  {np.max(norm(results.force_N,  axis=1)):.1f} N")
print(f"Max torque: {np.max(norm(results.torque_Nm, axis=1)):.3f} Nm")

In [ ]:
sigma_accel  = 1.0
sigma_gyro   = 1e-3
sigma_los    = 1.0
sigma_los_vel = 1.0
sigma_star   = 1e-3
sigma_doppler = 0.1
sigma_range  = 10.0

Q_ekf = np.zeros((13, 13))
Q_ekf[0:6,   0:6]   = Qd_from_accel_white(dt, sigma_accel)
Q_ekf[6:10,  6:10]  = np.eye(4) * 1e-6
Q_ekf[10:13, 10:13] = np.eye(3) * sigma_gyro**2 * dt


In [ ]:
# 3 orbiting satellites (same as main notebook)
sat0 = make_sat_arrs(results.t, altitude=100e3, raan=0,   aop=90, inc=90)
sat1 = make_sat_arrs(results.t, altitude=100e3, raan=90,  aop=94, inc=94)
sat2 = make_sat_arrs(results.t, altitude=100e3, raan=160, aop=70, inc=86)

# Beacons: stationary points on lunar surface
lz_pos = np.array([0., 0., R_MOON])                          # landing zone (directly below)
b2_pos = np.array([R_MOON * np.sin(np.radians(30)), 0., R_MOON * np.cos(np.radians(30))])  # 30 deg away

def make_beacon(pos, n):
    """Stationary beacon: fixed position, zero velocity."""
    return SatPosVel(r=np.tile(pos, (n, 1)), v=np.zeros((n, 3)))

beacon_lz = make_beacon(lz_pos, n)
beacon_b2 = make_beacon(b2_pos, n)

# Configurations: (name, [list of SatPosVel])
sat_configs = [
    ("3 Satellites",      [sat0, sat1, sat2]),
    ("2 Satellites",      [sat0, sat1]),
    ("1 Satellite",       [sat0]),
    ("Beacon at LZ",      [beacon_lz]),
    ("Beacon + 1 Sat",    [beacon_lz, sat0]),
    ("2 Beacons",         [beacon_lz, beacon_b2]),
]


In [ ]:
def make_suite(n_sats):
    return SensorSuite(sensors={
        "accelerometer":   accelerometer_sensor(sigma_accel),
        "gyroscope":       gyroscope_sensor(sigma_gyro),
        "laser_altimeter": laser_altimeter_sensor(sigma_los),
        "laser_velocity":  laser_velocity_sensor(sigma_los_vel),
        "star_tracker":    star_tracker_sensor(sigma_star),
        "doppler":         doppler_sensor(n_sats, sigma_doppler),
        "range_tracker":   sat_range_tracker_sensor(n_sats, sigma_range),
    })

sensor_frequencies = {
    "laser_altimeter": 1,
    "laser_velocity":  1,
    "star_tracker":    1,
    "doppler":         1,
    "range_tracker":   1,
}


In [ ]:
all_meas = {}

for cfg_name, sats in sat_configs:
    print(f"\n{cfg_name}")
    suite   = make_suite(len(sats))
    env_arr = generate_env(results, sim, sats)

    _, meas = generate_measurements(results.states, env_arr, suite)
    meas["accelerometer"] = measurements_noisy["accelerometer"]
    meas["gyroscope"]     = measurements_noisy["gyroscope"]

    all_meas[cfg_name] = {"meas": meas, "suite": suite, "env_arr": env_arr}

print("\nDone generating measurements.")

In [ ]:
config_results = []

for cfg_name, _ in sat_configs:
    entry   = all_meas[cfg_name]
    meas    = entry["meas"]
    suite   = entry["suite"]
    env_arr = entry["env_arr"]

    mu_arr    = np.zeros((n, 13))
    Sigma_arr = np.zeros((n, 13, 13))
    mu_arr[0]    = unitize_state(results.states[0])
    Sigma_arr[0] = np.eye(13)

    for i in tqdm(range(n - 1), desc=cfg_name, leave=False):
        accel_meas = meas["accelerometer"][i]
        gyro_meas  = meas["gyroscope"][i]

        mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
        mu_pred = unitize_state(mu_pred)

        for sensor, freq in sensor_frequencies.items():
            mu_pred, Sigma_pred = update_sensor(sensor, freq, mu_pred, Sigma_pred, env_arr[i], suite, meas, i)

        mu_arr[i + 1]    = mu_pred
        Sigma_arr[i + 1] = Sigma_pred

    pos_err = norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
    vel_err = norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)
    att_err = norm(mu_arr[:, 6:10] - results.states[:, 6:10], axis=1)

    config_results.append({
        "name": cfg_name, "mu_arr": mu_arr, "Sigma_arr": Sigma_arr,
        "pos_error": pos_err, "vel_error": vel_err, "att_error": att_err,
    })
    print(f"{cfg_name}: final pos error = {pos_err[-1]:.2f} m")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Liftoff EKF — Satellite Configuration Comparison', fontsize=14, fontweight='bold')
colors = plt.cm.tab10(np.linspace(0, 1, len(config_results)))

for i, r in enumerate(config_results):
    kw = dict(label=r['name'], color=colors[i], linewidth=2)
    axs[0].semilogy(results.t, r['pos_error'], **kw)
    axs[1].semilogy(results.t, r['vel_error'], **kw)
    axs[2].semilogy(results.t, r['att_error'], **kw)

for ax, (ylabel, title) in zip(axs, [
    ('Position Error (m)',  'Position Error'),
    ('Velocity Error (m/s)', 'Velocity Error'),
    ('Attitude Error',       'Attitude Error'),
]):
    ax.set_xlabel('Time (s)'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold'); ax.grid(True, alpha=0.3, which='both')

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.0, 0.5), fontsize=9, framealpha=0.9)
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()
